# RAG

In [ ]:
import time
start_time = time.time()

### Upload html documents from local folder

BSHTMLLoader: Strips all HTML immediately → tables become unformatted text  
Your custom function: Converts tables to markdown first → tables remain structured

In [ ]:
# Upload all files in folder "6k_filings"
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
from langchain_core.documents import Document

def html_to_clean_text(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    
    tables = soup.find_all('table')
    for table in tables:
        try:
            df = pd.read_html(StringIO(str(table)))[0]
            markdown = "\n" + df.to_markdown(index=False) + "\n"
            table.replace_with(soup.new_string(markdown))
        except:
            table.replace_with(soup.new_string(table.get_text(separator=' ', strip=True)))
    
    return soup.get_text(separator='\n', strip=True)

# Load all files
folder_path = "./6k_filings"
htm_files = list(Path(folder_path).glob("*.htm"))

documents = []
for doc_id, file_path in enumerate(htm_files):
    html_content = file_path.read_text(encoding='utf-8')
    clean_text = html_to_clean_text(html_content)
    
    doc = Document(
        page_content=clean_text,
        metadata={
            "source": str(file_path),
            "docid": doc_id  
        }
    )
    documents.append(doc)

### Split LangChain document objects into chunks that are as well LangChain document objects

Considered using MarkdownHeaderTextSplitter because used markdowns to clarify tables. Still, better RecursiveCharacterTextSplitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,    #  adds where each chunk starts in the original document
    separators=["\n\n", "\n", ". ", " ", ""]  # Splits on paragraphs, then lines, then sentences, then words, then characters
)
chunks = text_splitter.split_documents(documents)
print(f'Split {len(documents)} filings (documents) into {len(chunks)} chunks.' )

### Batch Embeddings
#### Prepare batches

In [ ]:
import json
from pathlib import Path

def create_batch_jsonl(
    chunks, 
    output_dir="batch_files",
    max_lines_per_file=10000
):
    """
    Create JSONL files for OpenAI batch embeddings with custom IDs and file limits
    
    Args:
        chunks: List of LangChain Document objects
        output_dir: Directory to save batch files
        max_lines_per_file: Maximum number of tasks per JSONL file
    
    Returns:
        List of created file paths
    """
    Path(output_dir).mkdir(exist_ok=True)
    batch_files = []
    
    # Split chunks into batches
    for batch_num in range(0, len(chunks), max_lines_per_file):
        batch_chunks = chunks[batch_num:batch_num + max_lines_per_file]
        output_file = f"{output_dir}/batch_for_embeddings_{batch_num // max_lines_per_file + 1}.jsonl"
        
        with open(output_file, 'w', encoding='utf-8') as f:
            for chunk in batch_chunks:
                # Create unique custom_id from metadata
                custom_id = (
                    str(chunk.metadata['docid']) + "_" + 
                    str(chunk.metadata['start_index'])
                )
                
                out_dict = {
                    "custom_id": custom_id,
                    "method": "POST",
                    "url": "/v1/embeddings",
                    "body": {
                        "model": "text-embedding-3-small",
                        "input": chunk.page_content
                    }
                }
                f.write(json.dumps(out_dict, ensure_ascii=False) + '\n')
        
        batch_files.append(output_file)
        print(f"Created {output_file} with {len(batch_chunks)} tasks")
    
    print(f"\nTotal: {len(batch_files)} batch file(s) created")
    return batch_files

# Create batch files
batch_files = create_batch_jsonl(chunks, max_lines_per_file=10000)

### Upload input file

In [ ]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [ ]:
from openai import OpenAI

client = OpenAI()
files = client.files.list()

In [ ]:
files.to_dict()['data']

In [ ]:
from glob import glob

batch_files = glob('./batch_files/batch_for_embeddings_*.jsonl')
batch_files

In [ ]:
from tqdm import tqdm
client = OpenAI()

my_batch_files_ids = []
for b_file in tqdm(batch_files):
    batch_input_file = client.files.create(
        file=open(b_file, "rb"), 
        purpose='batch'
    )
    my_batch_files_ids.append(batch_input_file.id)
    print(batch_input_file)

In [ ]:
my_batch_files_ids

In [ ]:
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.2f} seconds")